In [15]:
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)

# 1. Donn?es et d?coupage
df = pd.read_csv("../data/raw/aug_train.csv")
X = df.drop(columns=["target", "enrollee_id"])
y = df["target"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. Conversions
def convertir_experience(s):
    return pd.to_numeric(s.replace({"<1": "0", ">20": "21"}))

def convertir_last_new_job(s):
    return pd.to_numeric(s.replace({"never": "0", ">4": "5"}))

def convertir_numeriques(df_):
    df_ = df_.copy()
    df_["experience"] = convertir_experience(df_["experience"])
    df_["last_new_job"] = convertir_last_new_job(df_["last_new_job"])
    return df_

# 3. Pipeline de pr?paration
ohe = lambda: OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocess = ColumnTransformer([
    ("num_mediane", Pipeline([
        ("conversion", FunctionTransformer(convertir_numeriques, feature_names_out="one-to-one")),
        ("imputation", SimpleImputer(strategy="median"))]), ["experience", "last_new_job"]),
    ("cat_inconnu", Pipeline([
        ("imputation", SimpleImputer(strategy="constant", fill_value="Inconnu")),
        ("encodage", ohe())]), ["company_size", "company_type", "gender", "education_level"]),
    ("cat_na", Pipeline([
        ("imputation", SimpleImputer(strategy="constant", fill_value="Non applicable")),
        ("encodage", ohe())]), ["major_discipline"]),
    ("cat_freq", Pipeline([
        ("imputation", SimpleImputer(strategy="most_frequent")),
        ("encodage", ohe())]), ["enrolled_university"]),
    ("echelle", StandardScaler(), ["city_development_index", "training_hours"]),
    ("binaire", ohe(), ["relevent_experience"]),
    ("ville", ohe(), ["city"]),
])

# 4. Le mod?le final (v1), entra?n? sur le TRAIN seulement
modele_final = Pipeline([
    ("prep", preprocess),
    ("modele", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
modele_final.fit(X_train, y_train)

# 5. Validation crois?e et tableau des coefficients
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
}

noms = modele_final.named_steps["prep"].get_feature_names_out()
coefs = modele_final.named_steps["modele"].coef_[0]
tab = pd.DataFrame({"colonne": noms, "coef": coefs}).sort_values("coef")

print(X_train.shape, X_test.shape)
print(tab.iloc[0].to_dict(), tab.iloc[-1].to_dict())


(15326, 12) (3832, 12)
{'colonne': 'ville__city_city_7', 'coef': -1.66876383303097} {'colonne': 'ville__city_city_155', 'coef': 1.297959380036301}


In [16]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("job-change-prediction")

with mlflow.start_run(run_name="logreg_v1"):
    mlflow.log_param("modele", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("villes", "toutes")
    s = cross_validate(modele_final, X_train, y_train, cv=cv, scoring=scoring)
    for m in scoring:
        mlflow.log_metric(f"cv_{m}", s[f"test_{m}"].mean())

In [17]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

def enregistrer(nom, pipeline, params):
    with mlflow.start_run(run_name=nom):
        mlflow.log_params(params)
        s = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
        for m in scoring:
            mlflow.log_metric(f"cv_{m}", s[f"test_{m}"].mean())
            mlflow.log_metric(f"cv_{m}_std", s[f"test_{m}"].std())

preprocess_v2 = clone(preprocess).set_params(
    ville=OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, sparse_output=False))

rf_kw = dict(class_weight="balanced", random_state=42, n_jobs=-1)

essais = {
    "reference": (Pipeline([("prep", preprocess), ("modele", DummyClassifier(strategy="most_frequent"))]),
                  {"modele": "Dummy", "villes": "toutes"}),
    "logreg_v2": (Pipeline([("prep", preprocess_v2),
                            ("modele", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))]),
                  {"modele": "LogisticRegression", "villes": "regroupees_min100"}),
    "rf_defaut": (Pipeline([("prep", preprocess),
                            ("modele", RandomForestClassifier(n_estimators=200, **rf_kw))]),
                  {"modele": "RandomForest", "n_estimators": 200, "villes": "toutes"}),
    "rf_regularise": (Pipeline([("prep", preprocess),
                                ("modele", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, **rf_kw))]),
                      {"modele": "RandomForest", "n_estimators": 300, "max_depth": 8,
                       "min_samples_leaf": 20, "villes": "toutes"}),
}
for nom, (pipe, params) in essais.items():
    enregistrer(nom, pipe, params)

In [18]:
mlflow.search_runs(experiment_names=["job-change-prediction"])[
    ["tags.mlflow.runName", "metrics.cv_f1", "metrics.cv_recall"]]

,tags.mlflow.runName,metrics.cv_f1,metrics.cv_recall
0,rf_regularise,0.610100,0.752231
1,rf_defaut,0.584246,0.622978
2,logreg_v2,0.609826,0.742550
3,reference,0.000000,0.000000
4,logreg_v1,0.610751,0.740720
5,logreg_v1_final,NaN,NaN
6,rf_regularise,0.610100,0.752231
7,rf_defaut,0.584246,0.622978
8,logreg_v2,0.609826,0.742550
9,reference,0.000000,0.000000


In [19]:
y_pred = modele_final.predict(X_test)
y_proba = modele_final.predict_proba(X_test)[:, 1]
test_metrics = {
    "test_accuracy": accuracy_score(y_test, y_pred),
    "test_precision": precision_score(y_test, y_pred, zero_division=0),
    "test_recall": recall_score(y_test, y_pred, zero_division=0),
    "test_f1": f1_score(y_test, y_pred, zero_division=0),
    "test_roc_auc": roc_auc_score(y_test, y_proba),
    "test_pr_auc": average_precision_score(y_test, y_proba),
}

with mlflow.start_run(run_name="logreg_v1_final"):
    mlflow.log_params({"modele": "LogisticRegression", "class_weight": "balanced",
                       "villes": "toutes", "seuil": 0.5})
    mlflow.sklearn.log_model(
        modele_final, name="modele",
        skops_trusted_types=["__main__.convertir_numeriques", "numpy.dtype"],
    )
    mlflow.set_tag("statut", "officiel_evalue_sur_test")
    mlflow.log_metrics(test_metrics)


2026/11/02 20:04:52 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\bou6a\AppData\Local\Temp\tmpv7_9q2pq\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.1', 'skops==0.16.0']. Set logging level to DEBUG to see the full traceback. 
2026/11/02 20:04:52 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [20]:
runs = mlflow.search_runs(experiment_names=["job-change-prediction"])
runs = runs[runs["metrics.cv_f1"].notna()]
runs = runs.sort_values("start_time").drop_duplicates(
    subset="tags.mlflow.runName", keep="last"
)
runs[["tags.mlflow.runName", "metrics.cv_f1", "metrics.cv_recall"]]


,tags.mlflow.runName,metrics.cv_f1,metrics.cv_recall
5,logreg_v1,0.610751,0.740720
4,reference,0.000000,0.000000
3,logreg_v2,0.609826,0.742550
2,rf_defaut,0.584246,0.622978
1,rf_regularise,0.610100,0.752231
